In [1]:
import pandas as pd
df=pd.read_csv("iotSensor_data.csv")
print(df.shape)
df.head()

(10000, 15)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Unnamed: 8,Tool wear Failure,Heat dissipition failure,power failure,overstrain failure,random failure,Machine Failure
0,1,M14860,M,298.1,308.6,1551,42.8,0,NaN,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,NaN,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,NaN,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,NaN,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,NaN,0,0,0,0,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   UDI                       10000 non-null  int64  
 1   Product ID                10000 non-null  object 
 2   Type                      10000 non-null  object 
 3   Air temperature [K]       10000 non-null  float64
 4   Process temperature [K]   10000 non-null  float64
 5   Rotational speed [rpm]    10000 non-null  int64  
 6   Torque [Nm]               10000 non-null  float64
 7   Tool wear [min]           10000 non-null  int64  
 8   Unnamed: 8                0 non-null      float64
 9   Tool wear Failure         10000 non-null  int64  
 10  Heat dissipition failure  10000 non-null  int64  
 11  power failure             10000 non-null  int64  
 12  overstrain failure        10000 non-null  int64  
 13  random failure            10000 non-null  int64  
 14  Machine

In [4]:
df.isnull().sum()
print(df.dtypes)          

UDI                           int64
Product ID                   object
Type                         object
Air temperature [K]         float64
Process temperature [K]     float64
Rotational speed [rpm]        int64
Torque [Nm]                 float64
Tool wear [min]               int64
Unnamed: 8                  float64
Tool wear Failure             int64
Heat dissipition failure      int64
power failure                 int64
overstrain failure            int64
random failure                int64
Machine Failure               int64
dtype: object


In [56]:
df.duplicated().sum()

np.int64(0)

In [57]:
failure_cols = ['Tool wear Failure', 'Heat dissipition failure',
                'power failure', 'overstrain failure', 'random failure']

for col in failure_cols:
    failure = (df[col] == 1).sum()
    non_failure = (df[col] == 0).sum()
    print(f"{col}: Failure={failure}, Non-Failure={non_failure}")

# Check your new Failure column separately
print(f"\nFinal Failure column: Failure={( df['Failure'] == 1).sum()}, Non-Failure={(df['Failure'] == 0).sum()}")

Tool wear Failure: Failure=46, Non-Failure=9954
Heat dissipition failure: Failure=115, Non-Failure=9885
power failure: Failure=95, Non-Failure=9905
overstrain failure: Failure=98, Non-Failure=9902
random failure: Failure=19, Non-Failure=9981

Final Failure column: Failure=348, Non-Failure=9652


In [58]:
print(df.isnull().sum())

UDI                         0
Product ID                  0
Type                        0
Air temperature [K]         0
Process temperature [K]     0
Rotational speed [rpm]      0
Torque [Nm]                 0
Tool wear [min]             0
Machine failure             0
Tool wear Failure           0
Heat dissipition failure    0
power failure               0
overstrain failure          0
random failure              0
Failure                     0
dtype: int64


In [59]:
df = df.drop(columns=['UDI', 'Product ID'])

In [60]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df['Type']=le.fit_transform(df["Type"])

In [61]:
from sklearn.utils import resample

# Split FIRST on original data
X = df[['Type', 'Air temperature [K]', 'Process temperature [K]',
        'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]
y = df['Failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Upsample ONLY training data
train_data = pd.concat([X_train, y_train], axis=1)
failure = train_data[train_data['Failure'] == 1]
non_failure = train_data[train_data['Failure'] == 0]

failure_upsampled = resample(failure, replace=True,
                             n_samples=len(non_failure),
                             random_state=42)

train_balanced = pd.concat([non_failure, failure_upsampled])

X_train = train_balanced.drop(columns=['Failure'])
y_train = train_balanced['Failure']

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train value counts:\n{y_train.value_counts()}")

X_train: (15434, 6)
X_test: (2000, 6)
y_train value counts:
Failure
0    7717
1    7717
Name: count, dtype: int64


In [62]:
# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("Logistic Regression:")
print(classification_report(y_test, y_pred_lr))

# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest:")
print(classification_report(y_test, y_pred_rf))

Logistic Regression:
              precision    recall  f1-score   support

           0       0.99      0.81      0.89      1935
           1       0.12      0.78      0.21        65

    accuracy                           0.81      2000
   macro avg       0.56      0.80      0.55      2000
weighted avg       0.96      0.81      0.87      2000

Random Forest:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1935
           1       0.73      0.55      0.63        65

    accuracy                           0.98      2000
   macro avg       0.86      0.77      0.81      2000
weighted avg       0.98      0.98      0.98      2000



In [68]:
df = df.drop(columns=['Machine failure'])

In [69]:
X_all = df[['Type', 'Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]

df['Predicted_Failure'] = rf.predict(X_all)
df['Failure_Probability'] = rf.predict_proba(X_all)[:, 1]

def risk_level(prob):
    if prob >= 0.7:
        return 'High'
    elif prob >= 0.4:
        return 'Medium'
    else:
        return 'Low'

df['Risk_Level'] = df['Failure_Probability'].apply(risk_level)

df.to_csv('iot_final.csv', index=False)
print("Done!")
print(df.shape)

Done!
(10000, 15)


In [70]:
print(df.columns.tolist())
print(df.shape)
print(df[['Failure', 'Predicted_Failure', 'Failure_Probability', 'Risk_Level']].head(10))

['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Tool wear Failure', 'Heat dissipition failure', 'power failure', 'overstrain failure', 'random failure', 'Failure', 'Predicted_Failure', 'Failure_Probability', 'Risk_Level']
(10000, 15)
   Failure  Predicted_Failure  Failure_Probability Risk_Level
0        0                  0                  0.0        Low
1        0                  0                  0.0        Low
2        0                  0                  0.0        Low
3        0                  0                  0.0        Low
4        0                  0                  0.0        Low
5        0                  0                  0.0        Low
6        0                  0                  0.0        Low
7        0                  0                  0.0        Low
8        0                  0                  0.0        Low
9        0                  0                  0.0        Low


In [71]:
print(df['Predicted_Failure'].value_counts())

Predicted_Failure
0    9668
1     332
Name: count, dtype: int64


In [9]:
import pandas as pd
df=pd.read_csv("iot_final.csv")
print(df.shape)
df.head()

(10000, 15)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Tool wear Failure,Heat dissipition failure,power failure,overstrain failure,random failure,Failure,Predicted_Failure,Failure_Probability,Risk_Level
0,2,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,0,0.0,Low
1,1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,0,0.0,Low
2,1,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,0,0.0,Low
3,1,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,0,0.0,Low
4,1,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,0,0.0,Low


In [11]:
df['Failure'] = df['Failure'].astype(int)
df.to_csv('iot_final_v2.csv', index=False)
print("Done!")

Done!


In [12]:
print(df['Failure'].unique())

[0 1]
